In [1]:
# получить эксель по файла из папки
from glob import glob
import pandas as pd
import os
from lib.images import get_meta_from_image


def images_from_folder_to_excel(folder: str) -> str:
    i = 0
    everything = []

    char_name_id_map = {
        "isabella": 430
    }
    
    for image_path in glob(os.path.join(folder, "*.png")):
        #print(image_path)
        image_info = get_meta_from_image(image_path, char_name_id_map).model_dump()
        image_info["path"] = image_path
        image_info["cloths"].replace("undress state ", "")
        image_info["rating"].replace("questionable", "nudes")
        everything.append(image_info)


    df = pd.DataFrame(everything)
    df.to_excel(os.path.join(folder,"images.xlsx"))
    return "images.xlsx"

ffolder = "/Users/umaxfun/Downloads/Images"
#ffolder = "lib/test/img/img-upd"
images_from_folder_to_excel(ffolder)
#images_from_folder_to_excel("lib/test/img/img-upd")




'images.xlsx'

In [2]:
import uuid
from lib.content_models import ImageInfo
from typing import List
from typing import Dict
import pandas as pd


load = pd.read_excel(ffolder + "/images.xlsx").to_dict(orient="records")
#print(load.to_dict(orient="records"))

objs: List[ImageInfo] = []
paths: Dict[uuid.UUID, str] = {}

for x in load:
    objs.append(x)
    paths[x["id"]] = x["path"]
    print(x, x["path"])

{'Unnamed: 0': 0, 'id': 'd88ccc26-256f-4e2f-b140-75023ede82df', 'name': 'Isabella.json + 1_lounge_area_luxurious_sofas.json + 0_body_and_boots.json + 11_explicit_rating_Ass_Grab.json.png', 'hash': '0347c310bfe01b2fbfa76b0aa2e7b1c9ca8410095b13df2dae3a653e882768f9', 'character': 430, 'image': nan, 'image_blurred': nan, 'location': 'lounge area luxurious sofas', 'cloths': 'body and boots', 'rating': 'explicit', 'behavior': 'ass grab', 'prompt': '(((zPDXL, score_9, score_8_up, score_7_up, masterpiece, best quality, photo, photograph, source_realistic, raw, BREAK))) Woman, solo, 27 years old, Human, (Italian, (Lounge area with luxurious sofas, panoramic windows, ambient lights), ((fishnet bodysuit, knee-high stockings, high-heeled boots)), (((Restrained, experiencing vaginal penetration, , ass grab, top-down bottom-up, grabbing anothers ass, (rating_explicit:1.3), , ass grab, top-down bottom-up, grabbing anothers ass, (rating_explicit:1.3)))), dynamic angle, dramatic shadows, highly detaile

In [3]:
import requests
import os
import mimetypes

def get_file_id_by_name(api_url, token, folder_id, file_name):
    # Set the headers including the authentication token
    headers = {
        'Authorization': f'Bearer {token}'
    }
    
    # Make the request to get the file by name
    response = requests.get(
        f'{api_url}/files',
        headers=headers,
        params={'filter[title][_eq]': file_name, 'filter[folder][_eq]': folder_id}
    )
    
    # Check if the request was successful
    if response.status_code == 200:
        files = response.json().get('data', [])
        if files:
            return files[0]['id']
    else:
        # Handle errors (raise exception or return None)
        response.raise_for_status()

    return None

import requests

def upload_file_to_directus(api_url, token, file_path, folder_id, file_name):
    # Create a form data object
    form_data = {
        'folder': folder_id,
        'title': file_name,
    }

    # Open the file in binary mode
    with open(file_path, 'rb') as file:
        # Specify the content type of the file
        files = {'file': (file_path, file, mimetypes.guess_type(file_path)[0])}  # Replace 'application/octet-stream' with the appropriate MIME type if known

        # Set the headers including the authentication token
        headers = {
            'Authorization': f'Bearer {token}',
            # No need to set Content-Type for the form-data explicitly
        }

        # Make the request to upload the file
        response = requests.post(
            f'{api_url}/files',
            headers=headers,
            data=form_data,
            files=files
        )
    
    # Check if the request was successful
    if response.status_code == 200:
        # Parse the JSON response and return the file ID
        file_id = response.json().get('data', {}).get('id')
        return file_id
    else:
        # Handle errors (raise exception or return None)
        response.raise_for_status()

def patch_file_in_directus(api_url, token, file_path, file_id, file_name):
    # Create a form data object
    form_data = {
        'title': file_name,
    }

    # Open the file in binary mode
    with open(file_path, 'rb') as file:
        # Specify the content type of the file
        files = {'file': (file_path, file, mimetypes.guess_type(file_path)[0])}  # Replace 'application/octet-stream' with the appropriate MIME type if known

        # Set the headers including the authentication token
        headers = {
            'Authorization': f'Bearer {token}',
            # No need to set Content-Type for the form-data explicitly
        }

        # Make the request to upload the file
        response = requests.patch(
            f'{api_url}/files/'+file_id,
            headers=headers,
            data=form_data,
            files=files
        )
    
    # Check if the request was successful
    if response.status_code == 200:
        # Parse the JSON response and return the file ID
        file_id = response.json().get('data', {}).get('id')
        return file_id
    else:
        # Handle errors (raise exception or return None)
        response.raise_for_status()


# Usage example

In [4]:

from dotenv import load_dotenv
import importlib
import os
import mimetypes
from sqlmodel import create_engine, select
from sqlmodel import Session
from tempfile import mkdtemp
from wand.image import Image


load_dotenv()
api_url = os.environ.get('DIRECTUS_API_URL')
token = os.environ.get('DIRECTUS_TOKEN')


DB_URL = os.environ.get("DB_URL")

engine = create_engine(DB_URL)

def _magic_copy_from_dict(dict: dict, attached_model: ImageInfo):
    for key, value in dict.items():
        if key == "id":
            continue
        if hasattr(attached_model, key) == False:
            continue
        setattr(attached_model, key, value)

regular_folder_id = "6084434b-78ff-4976-a6e8-95d517698656"
blurred_folder_id = "65b17a30-f7a7-4907-be38-4d41c6032fdc"

TEMP_FOLDER_REGULAR = mkdtemp(prefix="uploader_reg_")
TEMP_FOLDER_BLURRED = mkdtemp(prefix="uploader_blurred")

def process_image(input_path):
    file_name = os.path.basename(input_path)
    temp_file_path_regular = os.path.join(TEMP_FOLDER_REGULAR, f"{os.path.splitext(file_name)[0]}.jpg")    
    temp_file_path_blurred = os.path.join(TEMP_FOLDER_BLURRED, f"{os.path.splitext(file_name)[0]}.jpg")
    # Convert the image
    width = 1000
    height = 1000
    quality = 85
    blur_radius = 80
    blur_sigma = 80

    with Image(filename=input_path) as img:
        img.resize(width, height)
        img.compression_quality = quality
        img.strip()
        img.blur(blur_radius, blur_sigma)
        img.format = 'jpeg'
        img.save(filename=temp_file_path_blurred)

    with Image(filename=input_path) as img:
        img.resize(width, height)
        img.compression_quality = quality
        img.strip()
        img.format = 'jpeg'
        img.save(filename=temp_file_path_regular)
        
    return temp_file_path_regular, temp_file_path_blurred
    

for x in objs: # type: ImageInfo
    print(paths[x["id"]])
    # first -- 
    is_new = False

    regular, blurred = process_image(paths[x["id"]])

    try:
        regular_id = get_file_id_by_name(api_url, token, regular_folder_id, x["name"])
        print(regular_id)
        if regular_id is None:
            regular_id = upload_file_to_directus(api_url, token, regular, regular_folder_id, x["name"])
            is_new = True
        else:
            patch_file_in_directus(api_url, token, regular, regular_id, x["name"])
        
        blurred_id = get_file_id_by_name(api_url, token, blurred_folder_id, x["name"])
        if blurred_id is None:
            blurred_id = upload_file_to_directus(api_url, token, blurred, blurred_folder_id, x["name"])
        else:
            patch_file_in_directus(api_url, token, blurred, blurred_id, x["name"])
        os.unlink(regular)
        os.unlink(blurred)
    except Exception as e:
        print(e)
        raise

    

    with Session(engine) as session:
        if is_new:
            obj = ImageInfo(**x)
        else:
            obj = session.exec(select(ImageInfo).where(ImageInfo.name == x["name"])).first()
            if obj is None:
                obj = ImageInfo(**x)
            else:
                _magic_copy_from_dict(x, obj)

        obj.image = regular_id
        obj.image_blurred = blurred_id
        session.add(obj)
        session.commit()
os.rmdir(TEMP_FOLDER_REGULAR)
os.rmdir(TEMP_FOLDER_BLURRED)


/Users/umaxfun/Downloads/Images/Isabella.json + 1_lounge_area_luxurious_sofas.json + 0_body_and_boots.json + 11_explicit_rating_Ass_Grab.json.png
1877b788-72f3-4250-aabe-f1d73a6f89b0
/Users/umaxfun/Downloads/Images/Isabella.json + 3_spa_center_description.json + 2_nightdress_and_home_slippers.json + 9_explicit_rating_anal_doggy_style.json.png
a3b7f7dc-9375-44f0-8ca9-804f4b7a9298
/Users/umaxfun/Downloads/Images/Isabella.json + 1_lounge_area_luxurious_sofas.json + 3_short_dress.json + 0_safe_rating_stand.json.png
63eb536e-a972-4c40-bd56-d04f71e21e68
/Users/umaxfun/Downloads/Images/Isabella.json + 1_lounge_area_luxurious_sofas.json + 5_underwear_corset_thong.json + 0_safe_rating_stand.json.png
f4773596-cf7e-450f-b97f-7fd928302aab
/Users/umaxfun/Downloads/Images/Isabella.json + 0_hotel_terrace_sea_view.json + 1_dress_and_shoes.json + 6_explicit_rating_Masturbating.json.png
d659d22f-6e3d-4bef-b77e-932a4d947ae9
/Users/umaxfun/Downloads/Images/Isabella.json + 2_private_massage_room.json + 3_s